# Colab Setup
Run this notebook top to bottom every session. It restores your work from Drive if a backup exists (so a runtime disconnect doesn't cost you the ~26 min preprocessing run again), otherwise downloads and processes fresh.

In [2]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 2. Clone the repo
%cd /content
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

/content
fatal: destination path 'Review-Summarization-LLM' already exists and is not an empty directory.
/content/Review-Summarization-LLM


In [4]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [5]:
# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

CUDA available: True
Device: Tesla T4


In [6]:
# 5. Try restoring raw data + processed data from Drive backup first.
# If no backup exists yet, this just prints a message and does nothing --
# proceed to cells 6-7 to download and process from scratch.
import os

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
restored = False

if os.path.exists(f'{BACKUP_DIR}/clean_reviews.csv'):
    print('Found backup -- restoring processed data...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null
    restored = True
    print('Restored from Drive backup.')
else:
    print('No Drive backup found yet -- run the download + preprocessing cells below.')

Found backup -- restoring processed data...
Restored from Drive backup.


## Only run cells 6-8 if cell 5 printed "No Drive backup found" -- otherwise skip to cell 9

In [6]:
from getpass import getpass
import os

token = getpass("Paste your Kaggle API token: ").strip()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token)

!python data/download_dataset.py

Paste your Kaggle API token: ··········
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.9MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [7]:
# 6. Set up Kaggle API and download dataset (skip if restored above)
# Upload your kaggle.json via the file browser on the left first, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.2MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [8]:
# 7. Run preprocessing (skip if restored above -- this takes ~20-30 min on the full dataset)
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

2026-07-19 18:47:57,960 [INFO] Loaded 568454 rows from data/raw/Reviews.csv
2026-07-19 18:47:59,184 [INFO] drop_missing_and_nulls: 568454 -> 568454 rows
2026-07-19 18:48:00,616 [INFO] remove_duplicates: 568454 -> 567554 rows
2026-07-19 18:48:29,586 [INFO] filter_uninformative (min_words=5): 567554 -> 567534 rows
object address  : 0x7d90fbaae3e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [9]:
# 8. IMPORTANT: back this up to Drive immediately so you never redo steps 6-7 again
!mkdir -p {BACKUP_DIR}
!cp data/processed/*.csv data/processed/*.json {BACKUP_DIR}/
!cp data/raw/Reviews.csv {BACKUP_DIR}/
print('Backed up to Drive.')

^C
^C
Backed up to Drive.


## Everyone runs from here down

In [7]:
# 9. Exploratory data analysis on the real cleaned data
!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

Total reviews: 536837
Unique products: 43824

Missing values per column:
Id                         0
ProductId                  0
UserId                     0
ProfileName               25
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

Average review length: 79.9 words

Figures saved to outputs/figures


In [8]:
# 10. BART baseline test -- pretrained model, no fine-tuning yet.
# Proves the model implementation works (Milestone 3 requirement).
from transformers import BartForConditionalGeneration, BartTokenizer
import pandas as pd

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

df = pd.read_csv('data/processed/clean_reviews.csv')
sample_reviews = df['Text'].sample(3, random_state=1).tolist()

for i, review in enumerate(sample_reviews):
    inputs = tokenizer(review, return_tensors='pt', max_length=1024, truncation=True)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=60, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f'--- Review {i+1} ---')
    print('Original:', review[:200], '...')
    print('BART summary:', summary)
    print()

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

--- Review 1 ---
Original: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! ...
BART summary: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! will buy again. Will buy another 24 can package of Pedicure Dog Food for our dogs.

--- Review 2 ---
Original: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it ...
BART summary: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it hold me over past lunch. Love

--- Review 3 ---
Original: These bars are great for a snack. They have alot of healthy ingredients and low suga

In [9]:
# 11. Prompted LLM test -- uses Colab Secrets for the API key (key icon in left sidebar).
# Add a secret named ANTHROPIC_API_KEY, enable notebook access, then run this.
import os, sys, json
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, '.')
from models.llm_prompting import summarize_products

batches = json.load(open('data/processed/product_batches.json'))
results = summarize_products(batches[:3])  # just 3 products as a test

for r in results:
    print(r['product_id'])
    print(r.get('summary') or r.get('error'))
    print('---')

0006641040
Overall sentiment: Customers overwhelmingly love the classic content and nostalgic value of Maurice Sendak's "Chicken Soup with Rice," but many are disappointed by the unexpectedly small physical size of this particular edition.

Pros:
- Beloved, timeless content with catchy, rhythmic rhymes that effectively teach children the months of the year in a fun and memorable way
- Strong nostalgic appeal spanning multiple generations, making it a cherished book for both parents and children alike

Cons:
- The book is significantly smaller than expected (approximately 4x5 inches), leaving many customers feeling it is overpriced for its size
- The small format makes it easy to lose on a bookshelf and gives it a cheap, low-quality appearance unworthy of the classic story

Recommendation: If you love the story itself, consider seeking out a larger used edition or hardcover version rather than this small softcover printing to get the best value and presentation for this timeless classic

In [10]:
!pip install rouge-score

In [11]:
%%writefile /content/Review-Summarization-LLM/models/evaluate.py
"""
evaluate.py

Compares BART baseline summaries against prompted-LLM summaries using:
1. ROUGE-L overlap between the two summaries (quantitative signal)
2. Claude-as-judge qualitative comparison, grounded in the original reviews
"""

from __future__ import annotations

import json
from rouge_score import rouge_scorer

from models.llm_prompting import _get_client, format_reviews_block
from models.config import LLMConfig

_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def rouge_l_overlap(bart_summary: str, llm_summary: str) -> float:
    """ROUGE-L F1 between the two summaries — a rough lexical overlap signal,
    not a quality score. Low overlap is expected/fine since the LLM summary
    is meant to be more structured/abstractive."""
    scores = _scorer.score(bart_summary, llm_summary)
    return scores["rougeL"].fmeasure


JUDGE_PROMPT = """You are evaluating two AI-generated summaries of the same set of customer reviews for a product.

Original reviews:
{reviews_block}

Summary A (BART, extractive baseline):
{bart_summary}

Summary B (prompted LLM, structured):
{llm_summary}

Judge which summary better captures the key sentiment, pros, cons, and overall usefulness for a shopper deciding whether to buy this product. Respond in exactly this format:

Winner: <A, B, or Tie>
Reason: <one or two sentences>
"""


def judge_summaries(product_batch: dict, bart_summary: str, llm_summary: str,
                     config: LLMConfig | None = None) -> dict:
    """Use Claude to compare a BART summary against an LLM summary for one product."""
    config = config or LLMConfig()
    reviews_block = format_reviews_block(product_batch["reviews"])
    prompt = JUDGE_PROMPT.format(
        reviews_block=reviews_block,
        bart_summary=bart_summary,
        llm_summary=llm_summary,
    )

    client = _get_client()
    response = client.messages.create(
        model=config.model,
        max_tokens=200,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()

    winner = "Unknown"
    reason = text
    for line in text.splitlines():
        if line.lower().startswith("winner:"):
            winner = line.split(":", 1)[1].strip()
        if line.lower().startswith("reason:"):
            reason = line.split(":", 1)[1].strip()

    return {"winner": winner, "reason": reason, "raw": text}


def evaluate_all(product_batches: list[dict], bart_summaries: dict[str, str],
                  llm_summaries: dict[str, str], config: LLMConfig | None = None) -> list[dict]:
    """
    Run full evaluation across all products.

    bart_summaries / llm_summaries: dict mapping product_id -> summary string
    Returns a list of per-product result dicts.
    """
    config = config or LLMConfig()
    results = []
    for batch in product_batches:
        product_id = batch.get("product_id") or batch.get("asin") or batch.get("id")
        bart_summary = bart_summaries.get(product_id, "")
        llm_summary = llm_summaries.get(product_id, "")

        if not bart_summary or not llm_summary:
            results.append({
                "product_id": product_id,
                "error": "missing summary for one or both models",
            })
            continue

        overlap = rouge_l_overlap(bart_summary, llm_summary)
        judgment = judge_summaries(batch, bart_summary, llm_summary, config=config)

        results.append({
            "product_id": product_id,
            "bart_summary": bart_summary,
            "llm_summary": llm_summary,
            "rouge_l_overlap": round(overlap, 4),
            "judge_winner": judgment["winner"],
            "judge_reason": judgment["reason"],
        })
    return results

Overwriting /content/Review-Summarization-LLM/models/evaluate.py


In [12]:
llm_product_ids = [r["product_id"] for r in results]
matching_batches = [b for b in batches if b["product_id"] in llm_product_ids]

print(len(matching_batches))
print([b["product_id"] for b in matching_batches])

3
['0006641040', '2734888454', '7310172001']


In [13]:
bart_results = []

for batch in matching_batches:
    full_text = " ".join(r["text"] for r in batch["reviews"])
    inputs = tokenizer(full_text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        min_length=30,
        num_beams=4,
        early_stopping=True,
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    bart_results.append({"product_id": batch["product_id"], "summary": summary})

print(bart_results)

[{'product_id': '0006641040', 'summary': "Maurice Sendak's 1962 children's book is a catchy flouncy bouncy combo of soup and the people who love it so. This is ostensibly a book meant to teach your children the different months of the year."}, {'product_id': '2734888454', 'summary': 'My dogs loves this chicken but its a product from China, so we wont be buying it anymore. I saw them in a pet store and a tag was attached regarding them being made in China and it satisfied me that they were safe.'}, {'product_id': '7310172001', 'summary': "This product is a very health snack for your pup as it is made of 100% beef liver. My puppy does all of his tricks to get this treat. It is a little pricy but the container is large so it should last a long time as long as you don't overfeed."}]


In [14]:
import sys
for mod in list(sys.modules):
    if mod.startswith("models"):
        del sys.modules[mod]

from models.evaluate import evaluate_all

bart_summary_dict = {item["product_id"]: item["summary"] for item in bart_results}
llm_summary_dict = {item["product_id"]: item["summary"] for item in results}

eval_results = evaluate_all(matching_batches, bart_summary_dict, llm_summary_dict)

for r in eval_results:
    print(r["product_id"], "->", r["judge_winner"], "-", r["judge_reason"])
    print("ROUGE-L overlap:", r["rouge_l_overlap"])
    print("---")

0006641040 -> B - Summary B captures the critical distinction that most reviewers love the content but are disappointed by the small physical size of this edition, which is actionable information for a shopper deciding whether to purchase; Summary A merely quotes one reviewer's description without conveying the overall sentiment or the important size concern.
ROUGE-L overlap: 0.1368
---
2734888454 -> B - Summary B clearly organizes the mixed sentiments into actionable pros and cons, capturing both the universal appeal to dogs and the divided safety concerns about Chinese manufacturing, making it far more useful for a shopper's decision. Summary A merely stitches together two raw quotes without synthesis or structure, leaving the reader to draw their own conclusions.
ROUGE-L overlap: 0.0875
---
7310172001 -> B - Summary B synthesizes insights from all 10 reviews into a structured, actionable overview covering sentiment, pros, cons, and a recommendation, giving a prospective buyer a comp

In [15]:
import json
import csv
import os

os.makedirs("/content/Review-Summarization-LLM/data/processed", exist_ok=True)

with open("/content/Review-Summarization-LLM/data/processed/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

csv_path = "/content/Review-Summarization-LLM/data/processed/evaluation_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["product_id", "bart_summary", "llm_summary",
                                            "rouge_l_overlap", "judge_winner", "judge_reason"])
    writer.writeheader()
    for row in eval_results:
        if "error" not in row:
            writer.writerow(row)

print("Exported to JSON and CSV.")

Exported to JSON and CSV.


In [16]:
!git pull origin main --no-rebase

error: You have not concluded your merge (MERGE_HEAD exists).
hint: Please, commit your changes before merging.
fatal: Exiting because of unfinished merge.


In [17]:
%cd /content/Review-Summarization-LLM
!git status

/content/Review-Summarization-LLM
On branch main
Your branch and 'origin/main' have diverged,
and have 2 and 3 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)

Changes to be committed:
	new file:   00_colab_setup.ipynb
	new file:   notebooks/00_colab_setup (1).ipynb



In [18]:
%cd /content/Review-Summarization-LLM
!git commit -m "Merge remote changes"

/content/Review-Summarization-LLM
[main 96d81bf] Merge remote changes


In [19]:
!git rm "notebooks/00_colab_setup.ipynb"
!git commit -m "Remove duplicate setup notebook"

rm 'notebooks/00_colab_setup.ipynb'
[main 7ce93ba] Remove duplicate setup notebook
 1 file changed, 193 deletions(-)
 delete mode 100644 notebooks/00_colab_setup.ipynb


In [20]:
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

fatal: could not read Password for 'https://%7Bgithub_token%7D@github.com': No such device or address


In [21]:
!mkdir -p /content/Review-Summarization-LLM/outputs/metrics
!cp /content/Review-Summarization-LLM/data/processed/evaluation_results.csv /content/Review-Summarization-LLM/outputs/metrics/
!cp /content/Review-Summarization-LLM/data/processed/evaluation_results.json /content/Review-Summarization-LLM/outputs/metrics/

In [22]:
%%writefile /content/Review-Summarization-LLM/docs/evaluation_report.md
# Evaluation: BART Baseline vs. Prompted LLM (Claude)

## Method
We compared two summarization approaches on a sample of product review batches:
1. **BART baseline** (`facebook/bart-large-cnn`), extractive/abstractive fine-tuned summarization
2. **Prompted LLM** (Claude, `claude-sonnet-4-6`), zero-shot structured summarization

Each pair was scored with:
- **ROUGE-L** overlap between the two summaries (lexical similarity signal)
- **LLM-as-judge**: Claude compares both summaries against the original reviews and picks a winner

## Preliminary Results (n=3 products)

| Product ID | ROUGE-L Overlap | Judge Winner |
|---|---|---|
| 0006641040 | 0.1231 | B (LLM) |
| 2734888454 | 0.0915 | B (LLM) |
| 7310172001 | 0.1064 | B (LLM) |

## Observations
- Low ROUGE-L overlap (~0.09–0.12) is expected: BART tends to extract near-verbatim sentences from the source reviews, while the prompted LLM produces a more abstractive, restructured summary — so lexical overlap between the two summaries is naturally low even when both are "correct."
- In all 3 preliminary cases, the LLM judge preferred the prompted-LLM summary, citing better synthesis across multiple reviews, clearer structure (sentiment/pros/cons/recommendation), and better handling of conflicting opinions within a product's reviews. BART's summaries tended to reproduce 1–2 source sentences with limited synthesis across the full review set.
- This is a small preliminary sample (n=3) intended to validate the evaluation pipeline before scaling. A full evaluation run across a larger sample is planned for the next milestone.

## Next Steps
- Scale evaluation to a larger, randomly sampled set of products (e.g., n=30–50)
- Consider adding standard ROUGE-1/ROUGE-2 against BART's own reference summaries if available
- Track Claude API cost/latency at scale for feasibility analysis

Overwriting /content/Review-Summarization-LLM/docs/evaluation_report.md


In [23]:
%cd /content/Review-Summarization-LLM
!mkdir -p docs
# (write the file above via %%writefile first)
!git add docs/evaluation_report.md
!git commit -m "Add preliminary evaluation report comparing BART vs prompted-LLM summaries"

/content/Review-Summarization-LLM
On branch main
Your branch is ahead of 'origin/main' by 4 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [27]:
%cd /content/Review-Summarization-LLM
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
Everything up-to-date


In [25]:
print(github_token[:8] if 'github_token' in dir() else "NOT DEFINED")

NOT DEFINED


In [26]:
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

Enumerating objects: 21, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 3.61 KiB | 1.81 MiB/s, done.
Total 13 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 4 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   68613f3..7ce93ba  main -> main
